# Stage 6: Streamlit Application Development

## Objective
Build an interactive Streamlit application that loads the five saved classification models, accepts a held-out test CSV, applies the fitted preprocessing pipeline, and displays evaluation results.

### Mandatory assignment features
1. CSV upload option (test data only)
2. Model selection dropdown
3. Evaluation metrics display
4. Confusion matrix or classification report

This implementation includes both a confusion matrix and a classification report, plus a downloadable predictions file.

### Important design
- Models and preprocessing are loaded from saved `.joblib` artifacts.
- The app does not train or refit models.
- The uploaded CSV must contain the actual target column `y` (`yes`/`no`) so metrics can be computed.
- `duration` is excluded from model inputs to prevent target leakage.
- Only held-out test data should be uploaded.

## 6.1 Required project structure

Before running the app, confirm that the project has this structure:

```text
ML_Assignment_2/
├── app.py
├── requirements.txt
├── data/
│   ├── raw/
│   │   └── bank.csv
│   └── processed/
│       └── test_processed.csv
├── models/
│   ├── preprocessor.joblib
│   ├── label_encoder.joblib
│   ├── logistic_regression.joblib
│   ├── decision_tree.joblib
│   ├── knn.joblib
│   ├── gaussian_nb.joblib
│   └── svm.joblib
└── notebooks/
    └── 06_streamlit_application.ipynb
```

The `app.py` file is placed at the repository root. The app expects the `models/` directory beside it.

## 6.2 Prepare a raw held-out test CSV

The app accepts raw test records, not the already encoded `test_processed.csv`. Recreate the same stratified 80:20 split used in Stage 3 and save only the raw test rows, including `y`.

Run this cell from the `notebooks/` directory. It uses the same `random_state=42` and `stratify=y` settings as Stage 3. Do not upload the full dataset.

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/raw/bank.csv", sep=";")

X = df.drop(columns=["y"])
y = df["y"]

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

raw_test_data = X_test_raw.copy()
raw_test_data["y"] = y_test_raw

raw_test_data.to_csv(
    "../data/processed/test_data.csv",
    index=False
)

print("Shape:", raw_test_data.shape)
print("Columns:", raw_test_data.columns.tolist())
print(raw_test_data["y"].value_counts())

Shape: (905, 17)
Columns: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']
y
no     801
yes    104
Name: count, dtype: int64


## 6.3 Create `requirements.txt`

Create `requirements.txt` in the project root. Pinning broad compatible major versions helps keep local and cloud environments consistent. If your BITS Virtual Lab uses different installed versions, record those versions and test before deployment.

In [ ]:
requirements = """streamlit
scikit-learn
numpy
pandas
matplotlib
seaborn
joblib
"""

with open("../requirements.txt", "w", encoding="utf-8") as file:
    file.write(requirements)

print(requirements)

## 6.4 Create the Streamlit application

The complete application source is provided in the companion file `app.py`.

Its main workflow is:

1. Load the saved preprocessor, label encoder, and five trained models.
2. Accept a CSV upload.
3. Validate that the target column `y` is present and contains `yes`/`no`.
4. Remove `y` and exclude `duration` from input features.
5. Apply the saved preprocessor.
6. Generate predictions and positive-class probabilities.
7. Calculate Accuracy, AUC, Precision, Recall, F1 Score, and MCC.
8. Display a confusion matrix and classification report.
9. Allow predictions to be downloaded as a CSV.

Do not paste a newly fitted preprocessing pipeline into the app. It must load the exact fitted transformer from Stage 3.

## 6.5 Run the application locally

Open a terminal at the project root (the directory containing `app.py`) and run:

```bash
pip install -r requirements.txt
streamlit run app.py
```

Streamlit will print a local URL, usually `http://localhost:8501`. Open it in a browser.

## 6.6 Test the application

Upload `data/processed/test_data.csv` using the file uploader.

Test each of the five model options and verify:

- The dataset preview is displayed.
- The selected model evaluates without errors.
- All six metrics are displayed.
- The confusion matrix is visible.
- The classification report is visible.
- The predictions CSV download works.

The uploaded test data must be the held-out 20% test split, not the full dataset and not the training data.

## 6.7 Common issues and checks

| Issue | Likely cause | Check |
|---|---|---|
| Model artifact not found | Incorrect folder structure | `models/` must be beside `app.py` |
| Missing required feature | CSV does not contain all expected input columns | Use the raw held-out test CSV |
| Unexpected target value | Target labels differ from `yes`/`no` | Inspect unique values in `y` |
| AUC error | Test upload contains only one target class | Use the complete held-out test split |
| Import error on deployment | Missing dependency | Add package to `requirements.txt` |
| Wrong feature shape | Different preprocessing or feature columns | Load the Stage 3 fitted preprocessor |

If the uploaded file has only one target class, ROC-AUC is undefined. Use the complete stratified test set for assignment evaluation.

## 6.8 Stage 6 completion checklist

- [ ] Raw held-out `test_data.csv` created
- [ ] `app.py` placed at project root
- [ ] `requirements.txt` created
- [ ] All model artifacts present in `models/`
- [ ] App starts locally without errors
- [ ] CSV upload works
- [ ] Model selection works for all five models
- [ ] Six evaluation metrics are displayed
- [ ] Confusion matrix is displayed
- [ ] Classification report is displayed
- [ ] Predictions download works
- [ ] Screenshot of local app captured

### Deliverables
- `app.py`
- `requirements.txt`
- `data/processed/test_data.csv`
- Local application screenshot

**Next stage:** Stage 7 – GitHub Repository Preparation and Streamlit Community Cloud Deployment.